# SIFLOW · run_7 · DiffusionGemma setup (SIFLOW-G)

Downloads DiffusionGemma-26B-A4B (~50GB fp16, fits A100-80GB) and tokenizes data in its tokenizer. Trains live in run_8. Optional reduced-support cache cell included.

**Hardware:** single A100-80GB, <12h. All artifacts are written to Google Drive so the next notebook resumes.

**Needs from previous notebooks:** run_0 passed

In [ ]:
# --- 1. Clone + install (run once per session) ---
REPO_URL = "https://github.com/kagtgi/siflow.git"   # <-- edit to your fork if needed
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# --- 2. Mount Drive + set artifact base (all sessions share this) ---
from siflow.utils import drive
drive.mount()
import os
os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
BASE = drive.base_dir()
print("artifacts ->", BASE)

> **Note:** confirm the documented HF class and mask token for `google/diffusiongemma-26B-A4B-it` and set `teacher.auto_class` / `teacher.mask_token` in `siflow/config/gemma.yaml` if needed. The MoE backbone uses eager attention by default.

In [ ]:
import torch
from siflow.teacher import GemmaTeacher
teacher = GemmaTeacher(dtype=torch.bfloat16)
print("vocab", teacher.vocab_size, "hidden", teacher.hidden_dim, "mask_id", teacher.mask_index)
ids = torch.full((1, 16), teacher.mask_index, device=teacher.device)
print("argmax:", teacher.logits(ids).argmax(-1)[0][:8].tolist())

In [ ]:
from siflow.data import build_token_chunks
tokz = teacher.tokenizer
n  = build_token_chunks(tokz, 256, 40_000, f"{BASE}/data/gemma_256.npy",
                        dataset="Skylion007/openwebtext", split="train")
nv = build_token_chunks(tokz, 256, 2_000, f"{BASE}/data/gemma_val.npy",
                        dataset="Skylion007/openwebtext", split="train", skip_seqs=40_000)
print("gemma chunks:", n, nv)

In [ ]:
# (OPTIONAL) reduced-support cache
# !python scripts/build_cache.py --config siflow/config/gemma.yaml \
#     --tokens {BASE}/data/gemma_256.npy --out {BASE}/cache/gemma --n 30000 --m 128 --batch 4

In [ ]:
# --- Save outputs to Drive (so the next notebook can resume) ---

print('saved to', BASE)